##Creation de la Base de donnée bronze_db

In [0]:
%sql
-- Création de la base de donnée
CREATE DATABASE IF NOT EXISTS bronze_db;

-- Affiche de la base de donnée
SHOW DATABASES;

In [0]:
%sql
-- utilisation de la base de donnée bronze_db
USE bronze_db;
 -- selectionné la base de donnée utilisée
 SELECT current_database()

In [0]:
%python
# voir le nombre de fichier dans le volumes nyc_taxi
display(
    dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi/bronze_data/")
      )

### Integration des données sources

In [0]:
# Lecture des fichiers parquet de façon automatique

df = spark.read.parquet(filePath)
# Affichage du dataframe df
display(df) 

In [0]:
# comptage du nombre de ligne dans le dataframe df et  affecté le resultat dans la variable count
count = df.count()
# Affichage du resultat
print("Le nombre de ligne du dataframe df est: ", count)

In [0]:
# Script d'ingestion_timestamp avec les condions suivants:
# Ne crée pas de nouvelle table si la table raw_trips exist
# Ne supprime pas les anciennes données de la table raw_trips
# Ajoute une colonne ingestion_timetamp dans la table
# s'il n'ya pas de nouveaux fichier parquet, ne fait rien
# Pour ajouter de nouvelle donnée de fichier parquet, compare le timestamp de la derniere modification dans le fichier parquet et la date de la derniere modification dans la table raw_trips
# Si le timestamp de la derniere modification dans le fichier parquet est plus recent que la date de la derniere modification dans la table raw_trips, ajoute les nouvelles données dans la table raw_trips
# Sinon, ne fait rien
# il faut que le type des variables doit correspondre au type de la colonne dans la table raw_trips
# Si la colonne n'existe pas, il faut la créer dans la table raw_trips
# Si la colonne existe, il faut la mettre à jour dans la table raw_trips


from pyspark.sql.functions import current_timestamp,col,substring_index
from pyspark.sql.types import TimestampType, StringType, DoubleType, IntegerType, StructType, StructField
from datetime import datetime
# Définir explicitement le schema des données
schema = StructType([
        StructField("VendorID", IntegerType(), True),
        StructField("tpep_pickup_datetime", TimestampType(), True),
        StructField("tpep_dropoff_datetime", TimestampType(), True),
        StructField("passenger_count", LongType(), True),
        StructField("trip_distance", DoubleType(), True),
        StructField("RatecodeID",LongType(), True),
        StructField("store_and_fwd_flag", StringType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", IntegerType(), True),
        StructField("payment_type", LongType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("extra", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("improvement_surcharge", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("congestion_surcharge", DoubleType(), True),
        StructField("Airport_fee", DoubleType(), True)
])
# Récupérer le nom du répertoire de données
source_folder = "/Volumes/workspace/default/nyc_taxi/bronze_data/"
filePath = source_folder + "*.parquet"
# Lister uniquement les fichiers parquets
files = dbutils.fs.ls(source_folder)

In [0]:
# la création de la colonne timpetamp dans le dataframe df
from pyspark.sql.functions import current_timestamp
df = df.withColumn("ingestion_timetamp", current_timestamp())
# Affichage du dataframe df
display(df)


In [0]:
#compte le nombre de ligne
count = df.count()
print("Le nombre de ligne du dataframe df est: ", count)

In [0]:
# stockage du dataframe dans la base de donnée bronze_db
df.write.format("delta").mode("append").saveAsTable("raw_trips")


In [0]:
%sql
-- Affichage de la table raw_trips
SELECT * FROM raw_trips LIMIT 10;


In [0]:
%sql
-- supprimer la table raw_trips
-- DROP TABLE raw_trips;


In [0]:
%sql
-- Renommer le nom du volume 
-- ALTER VOLUME default.bronze_files
-- RENAME TO nyc_taxi;

-----------------------
-- Path_tripdata_01 = /Volum es/workspace/default/nyc_taxi/bronze_data/yellow_tripdata_2024-01.parquet/

-- Path_zone_taxi = /Volumes/workspace/default/nyc_taxi/bronze_data/taxi_zone_lookup.csv/

In [0]:
%sql
